# Agent 5 — InBody Scan Extractor

### Agent Responsibility
> *"Extracts body composition metrics from an InBody scan image using Groq's vision model, then formats the output as a validated input for Agent 2."*

**Position in pipeline:** Agent 5 feeds into Agent 2, bypassing calculated BMR with the measured BMR from the InBody device.

In [ ]:
import re
import json
import base64
from pathlib import Path
from groq import Groq

In [ ]:
GROQ_API_KEY = "<YOUR_GROQ_API_KEY>"
client = Groq(api_key=GROQ_API_KEY)
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

## What Is an InBody Scan?

An **InBody scan** (bioelectrical impedance analysis) measures how electrical signals pass through body tissues to estimate composition. A standard InBody result sheet includes:

| Metric | Description |
|---|---|
| **Weight (kg)** | Total body mass |
| **Skeletal Muscle Mass (kg)** | Lean muscle attached to the skeleton; key indicator of anabolic progress |
| **Body Fat Mass (kg)** | Total fat tissue weight |
| **Body Fat %** | Fat mass as a percentage of total body weight |
| **BMI** | Body Mass Index (weight / height²); used here to derive height if not recorded |
| **BMR (kcal)** | Basal Metabolic Rate measured by the device — more accurate than formula-based estimates |
| **Visceral Fat Level** | Internal abdominal fat score (1–20 scale); health risk indicator |
| **Total Body Water (L)** | Total intracellular + extracellular water; reflects hydration and lean tissue |

Because the InBody device **directly measures BMR**, Agent 5 passes this value to Agent 2 instead of letting Agent 2 calculate it from the Mifflin-St Jeor formula.

In [ ]:
def extract_inbody_from_image(image_path: str) -> dict:
    """
    Read an InBody scan image, encode it as base64, and call the Groq vision
    model to extract all body composition metrics as a structured JSON dict.

    Parameters
    ----------
    image_path : str
        Absolute or relative path to the InBody scan image (JPEG or PNG).

    Returns
    -------
    dict with keys:
        weight_kg              : float | None
        skeletal_muscle_mass_kg: float | None
        body_fat_mass_kg       : float | None
        body_fat_percent       : float | None
        bmi                    : float | None
        bmr_kcal               : float | None
        visceral_fat_level     : int   | None
        total_body_water_L     : float | None
    """
    image_file = Path(image_path)
    if not image_file.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Determine MIME type from extension
    suffix = image_file.suffix.lower()
    mime_map = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}
    mime_type = mime_map.get(suffix, "image/jpeg")

    # Encode image as base64
    with open(image_file, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    extraction_prompt = """\
You are a medical data extraction assistant specialising in InBody body composition reports.

Carefully examine the InBody scan image and extract the following metrics.
Return ONLY a valid JSON object — no markdown, no explanations, no extra text.

Required JSON format:
{
  "weight_kg": <number or null>,
  "skeletal_muscle_mass_kg": <number or null>,
  "body_fat_mass_kg": <number or null>,
  "body_fat_percent": <number or null>,
  "bmi": <number or null>,
  "bmr_kcal": <number or null>,
  "visceral_fat_level": <integer or null>,
  "total_body_water_L": <number or null>
}

Rules:
- Use null for any metric that is not visible or not present in the image.
- Do NOT invent or estimate values — only extract what is explicitly printed.
- All numeric values must be plain numbers (no units, no strings).
- visceral_fat_level must be an integer between 1 and 20, or null.
"""

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:{mime_type};base64,{b64}"}
                    },
                    {
                        "type": "text",
                        "text": extraction_prompt
                    }
                ]
            }
        ],
        temperature=0
    )

    raw_content = response.choices[0].message.content.strip()

    # Strip markdown code fences if the model wraps the JSON
    raw_content = re.sub(r"^```[a-z]*\n?", "", raw_content)
    raw_content = re.sub(r"\n?```$", "", raw_content)

    try:
        extracted = json.loads(raw_content)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Vision model returned non-JSON output.\n"
            f"Raw response:\n{raw_content}\n"
            f"Parse error: {e}"
        )

    # Ensure all expected keys exist (fill missing ones with None)
    expected_keys = [
        "weight_kg", "skeletal_muscle_mass_kg", "body_fat_mass_kg",
        "body_fat_percent", "bmi", "bmr_kcal",
        "visceral_fat_level", "total_body_water_L"
    ]
    for key in expected_keys:
        extracted.setdefault(key, None)

    return extracted

In [ ]:
def inbody_to_agent2_input(
    inbody_data: dict,
    activity_level: str,
    goal: str,
    age: int,
    sex: str
) -> dict:
    """
    Convert extracted InBody metrics into the input format expected by Agent 2.

    Agent 2 requires: age, sex, height_cm, weight_kg, activity_level, goal.
    height_cm is not measured by InBody; it is back-calculated from BMI and
    weight when both are available: height_m = sqrt(weight_kg / bmi).

    Parameters
    ----------
    inbody_data    : dict  — output of extract_inbody_from_image()
    activity_level : str   — user-supplied activity level (passed through to Agent 2)
    goal           : str   — user-supplied goal (passed through to Agent 2)
    age            : int   — user-supplied age (not on InBody report)
    sex            : str   — 'male' or 'female' (not on InBody report)

    Returns
    -------
    dict with two top-level keys:
        'agent2_input' : dict ready to pass into Agent 2's validate_input()
        'extra_data'   : dict of additional InBody metrics for downstream agents
    """
    weight_kg = inbody_data.get("weight_kg")
    bmi       = inbody_data.get("bmi")

    # Derive height_cm from BMI and weight if both are present
    height_cm = None
    if weight_kg is not None and bmi is not None and bmi > 0:
        height_m  = (weight_kg / bmi) ** 0.5
        height_cm = round(height_m * 100, 1)

    agent2_input = {
        "age"            : age,
        "sex"            : sex,
        "height_cm"      : height_cm,
        "weight_kg"      : weight_kg,
        "activity_level" : activity_level,
        "goal"           : goal,
    }

    # Extra InBody metrics passed downstream (e.g. Agent 3 / Agent 4)
    extra_data = {
        "muscle_mass_kg"      : inbody_data.get("skeletal_muscle_mass_kg"),
        "body_fat_percent"    : inbody_data.get("body_fat_percent"),
        "body_fat_mass_kg"    : inbody_data.get("body_fat_mass_kg"),
        "visceral_fat_level"  : inbody_data.get("visceral_fat_level"),
        "measured_bmr_kcal"   : inbody_data.get("bmr_kcal"),
        "total_body_water_L"  : inbody_data.get("total_body_water_L"),
    }

    return {
        "agent2_input": agent2_input,
        "extra_data"  : extra_data,
    }

In [ ]:
def run_inbody_agent(
    image_path: str,
    activity_level: str,
    goal: str,
    age: int,
    sex: str
) -> dict:
    """
    Full Agent 5 pipeline.

    1. Calls the Groq vision model to extract InBody metrics from the image.
    2. Converts the extracted data into Agent 2's expected input format.
    3. Returns both the raw InBody data and the formatted Agent 2 input.

    Parameters
    ----------
    image_path     : str  — path to the InBody scan image
    activity_level : str  — e.g. 'moderate', 'active'
    goal           : str  — e.g. 'weight_loss', 'muscle_gain', 'maintenance'
    age            : int  — user's age in years
    sex            : str  — 'male' or 'female'

    Returns
    -------
    dict:
        'inbody_raw'   : raw extraction dict from the vision model
        'agent2_input' : validated-ready dict for Agent 2
        'extra_data'   : supplementary metrics for Agent 3 / Agent 4
    """
    print(f"[Agent 5] Extracting InBody data from: {image_path}")
    inbody_raw = extract_inbody_from_image(image_path)
    print("[Agent 5] Extraction complete.")

    result = inbody_to_agent2_input(
        inbody_data=inbody_raw,
        activity_level=activity_level,
        goal=goal,
        age=age,
        sex=sex
    )

    return {
        "inbody_raw"   : inbody_raw,
        "agent2_input" : result["agent2_input"],
        "extra_data"   : result["extra_data"],
    }

In [ ]:
# ---------------------------------------------------------------------------
# Sample test with mocked data (use when no image is available)
# This demonstrates the full output shape without making a vision API call.
# ---------------------------------------------------------------------------

mock_inbody = {
    "weight_kg"              : 82.5,
    "skeletal_muscle_mass_kg": 35.2,
    "body_fat_mass_kg"       : 18.4,
    "body_fat_percent"       : 22.3,
    "bmi"                    : 26.1,
    "bmr_kcal"               : 1820,
    "visceral_fat_level"     : 8,
    "total_body_water_L"     : 42.1
}

# Simulate the conversion step (no image or API call needed)
mock_result = inbody_to_agent2_input(
    inbody_data    = mock_inbody,
    activity_level = "moderate",
    goal           = "weight_loss",
    age            = 30,
    sex            = "male"
)

print("=" * 55)
print("MOCK InBody Raw Data")
print("=" * 55)
for k, v in mock_inbody.items():
    print(f"  {k:<30}: {v}")

print()
print("=" * 55)
print("Agent 2 Input (derived from InBody)")
print("=" * 55)
for k, v in mock_result["agent2_input"].items():
    print(f"  {k:<20}: {v}")

print()
print("=" * 55)
print("Extra Data (for Agent 3 / Agent 4)")
print("=" * 55)
for k, v in mock_result["extra_data"].items():
    print(f"  {k:<25}: {v}")

print()
print("Note: height_cm is back-calculated from BMI and weight.")
print(f"  sqrt({mock_inbody['weight_kg']} / {mock_inbody['bmi']}) * 100 = "
      f"{mock_result['agent2_input']['height_cm']} cm")

In [ ]:
# ---------------------------------------------------------------------------
# Live test: run the full pipeline on the real InBody scan image
# Uncomment and run this cell when inbodyscan.jpg is available.
# ---------------------------------------------------------------------------

# output = run_inbody_agent(
#     image_path     = "inbodyscan.jpg",
#     activity_level = "moderate",
#     goal           = "weight_loss",
#     age            = 30,
#     sex            = "male"
# )
#
# print("InBody Raw:", json.dumps(output["inbody_raw"], indent=2))
# print("Agent 2 Input:", json.dumps(output["agent2_input"], indent=2))
# print("Extra Data:", json.dumps(output["extra_data"], indent=2))

## Pipeline Integration

Agent 5 slots in at the top of the nutrition pipeline as an optional image-based input path:

```
inbodyscan.jpg
      |
      v
InBody Agent 5  (extracts metrics via Groq vision model)
      |
      |  agent2_input: {age, sex, height_cm, weight_kg, activity_level, goal}
      |  extra_data:   {muscle_mass, body_fat_percent, visceral_fat, measured_bmr}
      v
Agent 2  (validation, safety check, macro calculation)
         --> uses measured_bmr from InBody instead of formula-based estimate
      |
      v
Agent 3  (meal plan generation)
      |
      v
Agent 4  (output formatting / delivery)
```

### Key advantage
The InBody device measures BMR via bioelectrical impedance. Passing `measured_bmr_kcal` directly to Agent 2 is **more accurate** than the Mifflin-St Jeor formula, especially for individuals with atypical body composition (high muscle mass, high body fat, or significant water retention).

### Fields Agent 2 uses from `agent2_input`
| Field | Source |
|---|---|
| `age` | User-supplied (not on InBody report) |
| `sex` | User-supplied (not on InBody report) |
| `height_cm` | Back-calculated: `sqrt(weight_kg / bmi) * 100` |
| `weight_kg` | Directly from InBody scan |
| `activity_level` | User-supplied |
| `goal` | User-supplied |